In [8]:
import os
from dotenv import load_dotenv

# 明确指定 .env 文件路径（确保它在当前工作目录）
env_path = '.env'
print(f"当前工作目录: {os.getcwd()}")
print(f".env 文件存在吗？ {os.path.exists(env_path)}")

# 加载 .env
load_dotenv(dotenv_path=env_path)

# 打印所有环境变量（调试用）
print("\n=== 所有环境变量 ===")
for key, value in os.environ.items():
    if "API_KEY" in key:
        print(f"{key} = {value[:10]}...{value[-5:]}")  # 只显示前后，避免太长

# 单独检查 TAVILY_API_KEY
tavily_key = os.environ.get("TAVILY_API_KEY")
print(f"\nTAVILY_API_KEY = {tavily_key}")

当前工作目录: E:\devsoftware\code\jupyter\notebooks\第2章-LangChain入门
.env 文件存在吗？ False

=== 所有环境变量 ===

TAVILY_API_KEY = None


In [9]:
import os
print("当前工作目录:", os.getcwd())
# 加载环境变量
from dotenv import load_dotenv

load_dotenv()
print("TAVILY_API_KEY loaded:", os.environ.get("TAVILY_API_KEY"))

当前工作目录: E:\devsoftware\code\jupyter\notebooks\第2章-LangChain入门
TAVILY_API_KEY loaded: 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


# 1.定义工具

我们先通过一个例子来看看工具的作用。

假设我们需要一个能计算数学题的智能体。

但是，模型本身不擅长处理数学问题，如果涉及到精确计算，它往往无法给出准确答案


In [10]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
model = init_chat_model(
    model="qwen3.6-max-preview",
    model_provider="openai",
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url=os.getenv("DASHSCOPE_BASE_URL")
)
print(os.getenv("DASHSCOPE_API_KEY"))
print(os.getenv("TAVILY_API_KEY"))
agent = create_agent(model)
# agent = create_agent(
#     model="deepseek-chat",
#     system_prompt="你是一个算术奇才。"
# )

sk-d51d3ba02d6a44daa206a2943cf93c17
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


In [11]:
from langchain.messages import HumanMessage

question = HumanMessage(content="467的平方根是多少?")

for token, metadata in agent.stream(
    {"messages": [question]},
    stream_mode="messages"
):
    print(token.content, end="", flush=True)

467 不是完全平方数，因此它的平方根是一个**无理数**。

- **算术平方根（正值）**：`√467 ≈ 21.6102`（保留四位小数）
- **完整平方根**：`±21.6102`

更精确的值为：  
`√467 ≈ 21.61018278497426…`

💡 **说明**：
- 在中文数学语境中，“平方根”通常指正负两个值（±），而“算术平方根”专指正值。
- 日常计算或考试中，保留 2～4 位小数（如 21.61 或 21.6102）已足够。如需更高精度，可直接使用科学计算器输入 `√467`。

所以，通常涉及到精确的数学计算，我们都会通过定义工具用传统代码来计算，然后把工具交给模型去使用。


In [12]:
# 定义工具
from langchain.tools import tool

@tool
def square_root(x: float) -> float:
    """Calculate the square root of a number"""
    return x ** 0.5

In [13]:
# 通过装饰器定义工具名称
@tool("square_root")
def tool1(x: float) -> float:
    """Calculate the square root of a number"""
    return x ** 0.5

In [14]:
# 通过装饰器定义工具作用描述
@tool("square_root", description="Calculate the square root of a number")
def tool1(x: float) -> float:
    return x ** 0.5

In [15]:
# 通过自定义model来约束入参
from pydantic import BaseModel, Field
from typing import Literal


# 例如一个查询天气的tool
class WeatherInput(BaseModel):
    """查询天气的输入参数."""
    location: str = Field(description="City name or coordinates")
    units: Literal["celsius", "fahrenheit"] = Field(
        default="celsius",
        description="Temperature unit preference"
    )
    include_forecast: bool = Field(
        default=False,
        description="Include 5-day forecast"
    )

# 定义一个查询天气的tool
@tool(args_schema=WeatherInput)
def get_weather(location: str, units: str = "celsius", include_forecast: bool = False) -> str:
    """Get current weather and optional forecast."""
    temp = 22 if units == "celsius" else 72
    result = f"Current weather in {location}: {temp} degrees {units[0].upper()}"
    if include_forecast:
        result += "\nNext 5 days: Sunny"
    return result


工具调用方式与普通函数调用方式一致。


In [16]:
tool1.invoke({"x": 467})

21.61018278497431

In [17]:
get_weather.invoke({"location": "杭州", "include_forecast": True})

'Current weather in 杭州: 22 degrees C\nNext 5 days: Sunny'

# 2.添加工具到智能体


In [18]:
from langchain.agents import create_agent

agent = create_agent(
    model=model,
    tools=[tool1, get_weather],
    system_prompt="你是一个智能助手，你使用工具来解决用户问题。"
)

In [19]:
for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="467的平方根是多少?")]},
    stream_mode="messages"
):
    print(token.content, end="", flush=True)


21.61018278497431467的平方根大约是21.61018278497431。

In [20]:
for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="北京和杭州接下来几天天气如何?")]},
    stream_mode="messages"
):
    print(token.content, end="", flush=True)

Current weather in 北京: 22 degrees C
Next 5 days: SunnyCurrent weather in 杭州: 22 degrees C
Next 5 days: Sunny根据查询结果，两地近期天气如下：

🌤️ **北京**
- 当前气温：22°C
- 未来5天预报：以晴天为主

🌤️ **杭州**
- 当前气温：22°C
- 未来5天预报：以晴天为主

两地接下来几天天气都非常不错，阳光充足，气温舒适，很适合户外活动或出行。如需了解更详细的逐日温度变化、降水概率或穿衣建议，随时告诉我！

如果采用stream的updates模式，可以看到Agent执行的每一个步骤：

In [21]:
for chunk in agent.stream(
    {"messages": [HumanMessage(content="467、529的平方根是多少?")]},
    stream_mode="updates"
):
    for step, data in chunk.items():
        print(f"step: {step}")
        print(f"content: {data['messages'][-1].content_blocks}")
        print()


step: model
content: [{'type': 'tool_call', 'name': 'square_root', 'args': {'x': 467}, 'id': 'call_3096822d030a4107b7aa374c'}, {'type': 'tool_call', 'name': 'square_root', 'args': {'x': 529}, 'id': 'call_a48cbe0b3fec4e7b922f5ff1'}]

step: tools
content: [{'type': 'text', 'text': '21.61018278497431'}]

step: tools
content: [{'type': 'text', 'text': '23.0'}]

step: model
content: [{'type': 'text', 'text': '计算结果如下：\n\n* **467 的平方根**约为 **21.6102**（精确值：21.61018278497431）\n* **529 的平方根**是 **23**（精确值，因为 23² = 529）\n\n如有其他数学计算需求，随时告诉我！'}]



# 3.预定义Tool

LangChain中提供了很多预定义的Tool，方便我们使用。例如：


In [22]:
# 使用tavily作为web搜索工具
from langchain_tavily import TavilySearch

import getpass
import os

if not os.environ.get("TAVILY_API_KEY"):
    os.environ["TAVILY_API_KEY"] = getpass.getpass("Tavily API key:\n")

tool = TavilySearch(
    max_results=5,
    topic="general",
    # include_answer=False,
    # include_raw_content=False,
    # include_images=False,
    # include_image_descriptions=False,
    # search_depth="basic",
    # time_range="day",
    # include_domains=None,
    # exclude_domains=None
)

In [23]:
tool.invoke("蒸蚌是什么梗？")

{'query': '蒸蚌是什么梗？',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://www.huxiu.com/article/4820519.html',
   'title': '一只笨猫学习辨认萝卜和纸巾，竟成了近期最火的二创赛道？ - 虎嗅',
   'content': '# 一只笨猫学习辨认萝卜和纸巾，竟成了近期最火的二创赛道？. 萝卜。纸巾。蒸蚌！我敢说，但凡网瘾没那么重的朋友，估计都不知道这三个含猫量为0的词，居然是最近爆火的新猫梗。. 起初，大开门的主人还在致力于教它学会喵生中的第一个中文——纸巾。如果开门能在一堆干扰选项中指对答案，就会获得小零食的奖励，以及主人那一声惊喜的“真棒”！. 而当主人如法炮制，一连发布了好几个教它学“纸巾”的视频后，网友开始好奇，为啥开门妈这么执着于纸巾教学？. 如果真是这样，那就不免让人怀疑，开门会不会把鼠标当成纸巾叼着去给开门妈擦屁股。哪怕真教会了，等哪天实操的时候，开门妈在厕所里大喊：“纸巾”，开门也只能一直在客厅里拿那个爪子搭在纸巾上。. 然而，还没等“纸巾”这个题型吃透，下一个“萝卜”又接踵而至了。最近的一个视频里，开门头戴朱迪的警察小帽，神情却猫猫祟祟。只听开门妈又是萝卜又是纸巾的说着，它也随之不是这个就是那个地摸着。. 虽然，为数不多的时候，开门聪明的智商会重新占领高地。就比如最近一期学习认识“米老鼠”的视频里，它前半段的答题就非常之完美。. 就连大家也都给唬住了，以为真让它学会了，嘴里还不停叫嚷着“博主塌房了”、“不信，这是AI”。. 这毫不拖泥带水的稍息动作一出，网友就了然了，原来，它依旧蒙题、依旧乱猜。好好好，既然如此那我问你，何棒之有？. 就这样，主人巧施连环计，开门误成大笨猫。网上随便溜达一圈，都有网友指着嘎嘎笑它智力不详。. 先不说大部分猫的大脑皮层如上图一样，光滑到几乎没有褶皱，单说开门这个个例。你可以说它算不上考研的料，但察言观色这一块，人情世故这一块，它完全是考公的好“猫”才。. 看这犹豫试探的小表情，没做对的时候还会撒娇两下，试图“萌”混过关。. 又或许是开门妈那一声声“蒸蚌”听着太洗脑，让人一看到视频就跟那个巴浦洛夫的狗听到铃声开始流口水一样，

In [24]:
# 创建智能体，使用预定义工具tavily
# agent = create_agent(
#     model="deepseek-chat",
#     tools=[tool],
#     system_prompt="你是一个智能助手，你使用工具来解决用户问题。"
# )
agent = create_agent(
    model=model,
    tools=[tool],
    system_prompt="你是一个智能助手，你需要使用工具来解决用户问题。"
)

In [25]:
# 调用工具
for chunk in agent.stream(
    {"messages": [HumanMessage(content="你是什么模型？什么是蒸蚌")]},
    stream_mode="updates"
):
    for step, data in chunk.items():
        print(f"step: {step}")
        print(f"content: {data['messages'][-1].content_blocks}")
        print()

step: model
content: [{'type': 'tool_call', 'name': 'tavily_search', 'args': {'query': '蒸蚌 是什么菜 或 含义'}, 'id': 'call_aa128b77c59c46afb7e352ee'}]

step: tools
content: [{'type': 'text', 'text': '{"query": "蒸蚌 是什么菜 或 含义", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://zh.hinative.com/questions/15904858", "title": "\\"我蒸蚌\\"是什么意思？ -关于中文(繁体 - HiNative", "content": "蒸蚌是「真棒」的意思因為“蒸蚌” 跟“真棒” 的音一模一樣，所以有一些人在寫「真棒」的時候，會故意用蒸蚌兩字，然後只是因為這樣子看起來很有趣 · 中文(简体)", "score": 0.709684, "raw_content": null}, {"url": "https://www.instagram.com/reel/DS9r6NzCSKV/", "title": "隨便指一指，聽到蒸蚌代表指對了   #貓狗吐槽主人#貓狗電台#米老鼠 ...", "content": "所以你現在知道米老鼠是什麼意思了嗎? 我知道, 是少悉的意思啊. 那真棒的意思呢? 真棒是吃東西的意思. 米老鼠. 真棒.", "score": 0.5826579, "raw_content": null}, {"url": "https://baike.baidu.com/item/%E8%92%9C%E9%A6%99%E8%92%B8%E8%9A%8C%E4%BB%94/7524839", "title": "蒜香蒸蚌仔_百度百科", "content": "蒜香蒸蚌仔是一道以象鼻蚌幼仔为主料的海鲜蒸菜，选用带壳重约850克的蚌仔10个，搭配水发粉丝100克制成。菜品以蚌肉鲜嫩弹滑、蒜香浓郁清甜为特色，兼具营养价值与易消化", "score": 0.5423064, "raw_content":

invoke调用测试

In [26]:
# 调用工具
response=agent.invoke(
        {"messages": [HumanMessage(content="你是什么模型？什么是蒸蚌")]},
)
for message in response['messages']:
    message.pretty_print()

================================ Human Message =================================

你是什么模型？什么是蒸蚌
================================== Ai Message ==================================
Tool Calls:
  tavily_search (call_a1bb08d24c6640c68d4837a2)
 Call ID: call_a1bb08d24c6640c68d4837a2
  Args:
    query: 蒸蚌 是什么
================================= Tool Message =================================
Name: tavily_search

{"query": "蒸蚌 是什么", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://zh.hinative.com/questions/15904858", "title": "\"我蒸蚌\"是什么意思？ -关于中文(繁体 - HiNative", "content": "我蒸蚌蒸蚌是「真棒」的意思因為“蒸蚌” 跟“真棒” 的音一模一樣，所以有一些人在寫「真棒」的時候，會故意用蒸蚌兩字，然後只是因為這樣子看起來很有趣 的", "score": 0.99990535, "raw_content": null}, {"url": "https://www.instagram.com/reel/DS9r6NzCSKV/", "title": "隨便指一指，聽到蒸蚌代表指對了   #貓狗吐槽主人#貓狗電台#米老鼠 ...", "content": "所以你現在知道米老鼠是什麼意思了嗎? 我知道, 是少悉的意思啊. 那真棒的意思呢? 真棒是吃東西的意思. 米老鼠. 真棒.", "score": 0.99783427, "raw_content": null}, {"url": "https://www.sohu.com/a/972125782_25885

自定义tavily工具

In [27]:
#先使用官方客户端做初始化
tavily=TavilySearch(
    max_results=5,
    topic="general"
)
#自己封装为tool
def web_search(query: str) -> dict:
    """search the web for information"""
    return tavily.invoke({"query": query})

定义结构化输出实体

In [28]:
from pydantic import BaseModel, Field

# agent 回答引用的网页信息
class Reference(BaseModel):
    title:str=Field(description="the title of the web page cited in the answer")
    url:str=Field(description="the url of the web page cited in the answer")

# agent 的回答
class AnswerInfo(BaseModel):
    answer:str=Field(description="the final answer for the user")
    reference:list[Reference]=Field(description="The web page cited in the answer")

In [32]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
model = init_chat_model(
    model="deepseek-v4-flash",
    model_provider="openai",
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url=os.getenv("DASHSCOPE_BASE_URL")
)
# 创建智能体，使用预定义工具tavily
agent = create_agent(
    model="deepseek-chat",
    tools=[web_search],
    system_prompt="你是一个智能助手，会使用工具来解决问题",
    response_format=AnswerInfo
)
# agent = create_agent(
#     model=model,
#     tools=[web_search],
#     system_prompt="你是一个智能助手，会使用工具来解决问题",
#     response_format=AnswerInfo
# )

In [33]:
# 调用agent
response=agent.invoke(
    {"messages": [HumanMessage(content="我要验牌是什么意思")]},
)
# 获取结构化输出
print(response)


{'messages': [HumanMessage(content='我要验牌是什么意思', additional_kwargs={}, response_metadata={}, id='345d74fe-2bfc-49e2-99d5-e7680cd9c54c'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 39, 'prompt_tokens': 438, 'total_tokens': 477, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 384}, 'prompt_cache_hit_tokens': 384, 'prompt_cache_miss_tokens': 54}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_058df29938_prod0820_fp8_kvcache_20260402', 'id': 'a48cc7fa-b24d-49c7-8b9e-99d1ea18a975', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019ddedd-3345-7df3-bab8-27af487aded8-0', tool_calls=[{'name': 'web_search', 'args': {'query': '验牌 什么意思'}, 'id': 'call_00_K0FD7pLykDWATPHZrB26BgbE', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 438, 'output_tokens': 39, 'total_tokens': 477, 'input_token_det